## Step 1: Reshape matches into one row per team per match
Melts each match into two rows (home perspective, away perspective) and computes `rolling_form_pts` — points from each team's last 5 matches within the same season, shifted by 1 to avoid leaking the current match's own result.

In [1]:
import pandas as pd

clean_matches = pd.read_csv("../data/processed/clean_matches_multi_season.csv", parse_dates=["utcDate"], dtype={"season": str})
clean_matches = clean_matches.sort_values("utcDate").reset_index(drop=True)

home_rows = clean_matches[["utcDate", "matchday", "season", "id", "home_team_name", "home_goals", "away_goals", "winner"]].copy()
home_rows.columns = ["utcDate", "matchday", "season", "match_id", "team", "goals_scored", "goals_conceded", "winner"]
home_rows["is_home"] = True
home_rows["result"] = home_rows["winner"].map({"HOME_TEAM": "WIN", "AWAY_TEAM": "LOSS", "DRAW": "DRAW"})

away_rows = clean_matches[["utcDate", "matchday", "season", "id", "away_team_name", "away_goals", "home_goals", "winner"]].copy()
away_rows.columns = ["utcDate", "matchday", "season", "match_id", "team", "goals_scored", "goals_conceded", "winner"]
away_rows["is_home"] = False
away_rows["result"] = away_rows["winner"].map({"HOME_TEAM": "LOSS", "AWAY_TEAM": "WIN", "DRAW": "DRAW"})

team_matches = pd.concat([home_rows, away_rows]).sort_values(["team", "utcDate"]).reset_index(drop=True)
team_matches.drop(columns="winner", inplace=True)
team_matches["season"] = team_matches["season"].astype(str)

points_map = {"WIN": 3, "DRAW": 1, "LOSS": 0}
team_matches["points"] = team_matches["result"].map(points_map)

team_matches["rolling_form_pts"] = (
    team_matches.groupby(["team", "season"])["points"]
    .transform(lambda x: x.shift(1).rolling(window=5, min_periods=1).sum())
)

## Step 2: Streaks, scoring averages, and per-match context features
Win/loss streaks, expanding average goals scored/conceded, goal difference, clean sheet rate, days since last match, and home/away win percentages — all shifted by 1 so no feature ever uses information from the match it's predicting.

> **Note:** `home_win_pct`/`away_win_pct` are computed and saved here but are not used by the trained model (see `07_machine_leaning.ipynb`) — they were excluded during training. Left in the saved CSV for completeness/potential future use.

In [2]:
def compute_streak(results, target):
    streak = 0
    streaks = []
    for r in results:
        streaks.append(streak)  # streak BEFORE this match
        streak = streak + 1 if r == target else 0
    return streaks

team_matches["win_streak"] = team_matches.groupby(["team", "season"])["result"].transform(
    lambda x: compute_streak(x, "WIN")
)
team_matches["loss_streak"] = team_matches.groupby(["team", "season"])["result"].transform(
    lambda x: compute_streak(x, "LOSS")
)

team_matches["avg_goals_scored"] = (
    team_matches.groupby(["team", "season"])["goals_scored"]
    .transform(lambda x: x.shift(1).expanding().mean())
)
team_matches["avg_goals_conceded"] = (
    team_matches.groupby(["team", "season"])["goals_conceded"]
    .transform(lambda x: x.shift(1).expanding().mean())
)

team_matches[team_matches["team"] == "AFC Bournemouth"][
    ["utcDate", "result", "points", "rolling_form_pts", "win_streak", "loss_streak", "avg_goals_scored", "avg_goals_conceded"]
].head(10)

team_matches["goal_difference"] = team_matches["goals_scored"] - team_matches["goals_conceded"]

team_matches["clean_sheet"] = (team_matches["goals_conceded"] == 0).astype(int)
team_matches["clean_sheet_pct"] = (
    team_matches.groupby(["team", "season"])["clean_sheet"]
    .transform(lambda x: x.shift(1).expanding().mean())
)

team_matches["days_since_last_match"] = (
    team_matches.groupby(["team", "season"])["utcDate"]
    .diff()
    .dt.days
)

team_matches["is_win"] = (team_matches["result"] == "WIN").astype(int)

team_matches["home_win_pct"] = (
    team_matches[team_matches["is_home"]]
    .groupby(["team", "season"])["is_win"]
    .transform(lambda x: x.shift(1).expanding().mean())
)

team_matches["away_win_pct"] = (
    team_matches[~team_matches["is_home"]]
    .groupby(["team", "season"])["is_win"]
    .transform(lambda x: x.shift(1).expanding().mean())
)

team_matches[team_matches["team"] == "AFC Bournemouth"][
    ["utcDate", "is_home", "goal_difference", "clean_sheet_pct", "days_since_last_match", "home_win_pct", "away_win_pct"]
].head(10)

,utcDate,is_home,goal_difference,clean_sheet_pct,days_since_last_match,home_win_pct,away_win_pct
0,2023-08-12 14:00:00+00:00,True,0,NaN,NaN,NaN,NaN
1,2023-08-19 14:00:00+00:00,False,-2,0.000000,7.0,NaN,NaN
2,2023-08-26 11:30:00+00:00,True,-2,0.000000,6.0,0.0,NaN
3,2023-09-02 14:00:00+00:00,False,0,0.000000,7.0,NaN,0.0
4,2023-09-17 13:00:00+00:00,True,0,0.000000,14.0,0.0,NaN
5,2023-09-24 13:00:00+00:00,False,-2,0.200000,7.0,NaN,0.0
6,2023-09-30 14:00:00+00:00,True,-4,0.166667,6.0,0.0,NaN
7,2023-10-07 14:00:00+00:00,False,-3,0.142857,7.0,NaN,0.0
8,2023-10-21 14:00:00+00:00,True,-1,0.125000,14.0,0.0,NaN
9,2023-10-28 14:00:00+00:00,True,1,0.111111,7.0,0.0,NaN


## Step 3: Head-to-head history
Builds each team's win rate against a specific opponent, across all historical seasons (not reset per season) — again shifted by 1 to exclude the current match.

In [3]:
h2h_records = []

for _, match in clean_matches.iterrows():
    h2h_records.append({"team": match["home_team_name"], "opponent": match["away_team_name"],
                         "utcDate": match["utcDate"], "goals_for": match["home_goals"], "goals_against": match["away_goals"]})
    h2h_records.append({"team": match["away_team_name"], "opponent": match["home_team_name"],
                         "utcDate": match["utcDate"], "goals_for": match["away_goals"], "goals_against": match["home_goals"]})

h2h_df = pd.DataFrame(h2h_records).sort_values(["team", "opponent", "utcDate"]).reset_index(drop=True)

h2h_df["h2h_win"] = (h2h_df["goals_for"] > h2h_df["goals_against"]).astype(int)

h2h_df["h2h_win_pct"] = (
    h2h_df.groupby(["team", "opponent"])["h2h_win"]
    .transform(lambda x: x.shift(1).expanding().mean())
)

h2h_df[h2h_df["h2h_win_pct"].notna()].head(10)

,team,opponent,utcDate,goals_for,goals_against,h2h_win,h2h_win_pct
1,AFC Bournemouth,Arsenal FC,2024-05-04 11:30:00+00:00,0,3,0,0.000000
2,AFC Bournemouth,Arsenal FC,2024-10-19 16:30:00+00:00,2,0,1,0.000000
3,AFC Bournemouth,Arsenal FC,2025-05-03 16:30:00+00:00,2,1,1,0.333333
4,AFC Bournemouth,Arsenal FC,2026-01-03 17:30:00+00:00,2,3,0,0.500000
5,AFC Bournemouth,Arsenal FC,2026-04-11 11:30:00+00:00,2,1,1,0.400000
7,AFC Bournemouth,Aston Villa FC,2024-04-21 14:00:00+00:00,1,3,0,0.000000
8,AFC Bournemouth,Aston Villa FC,2024-10-26 14:00:00+00:00,1,1,0,0.000000
9,AFC Bournemouth,Aston Villa FC,2025-05-10 16:30:00+00:00,0,1,0,0.000000
10,AFC Bournemouth,Aston Villa FC,2025-11-09 14:00:00+00:00,0,4,0,0.000000
11,AFC Bournemouth,Aston Villa FC,2026-02-07 15:00:00+00:00,1,1,0,0.000000


## Step 4: Attach head-to-head win rate onto the main dataset

In [4]:
team_matches = team_matches.merge(
    h2h_df[["team", "utcDate", "h2h_win_pct"]],
    on=["team", "utcDate"],
    how="left"
)

season_points = (
    team_matches.groupby(["team", "season"])
    .agg(total_points=("points", "sum"), total_goal_diff=("goal_difference", "sum"))
    .reset_index()
)
season_points["season"] = season_points["season"].astype(str)

## Step 5: Compute final league standings per season
Ranks teams within each season by points, then goal difference (Premier League tie-break rule), using each season's real start date to establish chronological order.

In [5]:
# --- Compute final standings per season ---
season_points = (
    team_matches.groupby(["team", "season"])
    .agg(total_points=("points", "sum"), total_goal_diff=("goal_difference", "sum"))
    .reset_index()
)

# Get each season's real start date to establish chronological order
# (season column is the messy raw string, but we can recover order via each season's earliest match date)
season_order = (
    clean_matches.groupby("season")["utcDate"].min()
    .sort_values()
    .reset_index()
    .rename(columns={"utcDate": "season_start"})
)
season_order["season_rank"] = range(len(season_order))

season_points = season_points.merge(season_order, on="season")

# Final position within each season: rank by points, then goal difference (PL tie-break rule)
season_points = season_points.sort_values(
    ["season", "total_points", "total_goal_diff"], ascending=[True, False, False]
)
season_points["final_position"] = season_points.groupby("season").cumcount() + 1

season_points = season_points.sort_values(["team", "season_rank"]).reset_index(drop=True)

## Step 6: Derive a defensible default for promoted teams
Newly-promoted teams have no prior top-flight season to look back on — rather than guessing a number, this computes the actual historical average finishing position of promoted teams already in the dataset.

In [6]:
# --- Historical average finishing position of newly-promoted teams (our defensible default) ---
season_points["prev_season_rank"] = season_points["season_rank"] - 1

promoted_mask = ~season_points.apply(
    lambda row: ((season_points["team"] == row["team"]) &
                 (season_points["season_rank"] == row["prev_season_rank"])).any(),
    axis=1
)
# Exclude each dataset's very first season (nobody has "previous" data, not a real promotion signal)
promoted_mask &= season_points["season_rank"] > 0

promoted_avg_position = season_points.loc[promoted_mask, "final_position"].mean()
print(f"Historical average final position for newly-promoted teams: {promoted_avg_position:.2f}")
print(f"(based on {promoted_mask.sum()} promoted-team-seasons in the dataset)")

Historical average final position for newly-promoted teams: 16.17
(based on 6 promoted-team-seasons in the dataset)


## Step 7: Build the `prior_season_position` feature
Looks up each team's actual previous-season finish (or the promoted-team fallback if they have none) and attaches it as a static per-team-per-season feature.

In [7]:
# --- Build (team, season) -> prior season's final position ---
position_lookup = season_points.set_index(["team", "season_rank"])["final_position"].to_dict()

def get_prior_position(row):
    key = (row["team"], row["season_rank"] - 1)
    return position_lookup.get(key, promoted_avg_position)

season_points["prior_season_position"] = season_points.apply(get_prior_position, axis=1)

# Attach onto team_matches (static per team+season)
team_matches = team_matches.merge(
    season_points[["team", "season", "prior_season_position"]],
    on=["team", "season"],
    how="left"
)

team_matches[["team", "season", "prior_season_position"]].drop_duplicates().sort_values(["season", "prior_season_position"]).head(10)

,team,season,prior_season_position
0,AFC Bournemouth,2023,16.166667
114,Arsenal FC,2023,16.166667
228,Aston Villa FC,2023,16.166667
342,Brentford FC,2023,16.166667
456,Brighton & Hove Albion FC,2023,16.166667
570,Burnley FC,2023,16.166667
646,Chelsea FC,2023,16.166667
760,Crystal Palace FC,2023,16.166667
874,Everton FC,2023,16.166667
988,Fulham FC,2023,16.166667


## Step 8: Ensure the output folder exists

In [8]:
import os
os.makedirs("../data/features", exist_ok=True)

## Step 9: Save the final feature set

In [9]:
team_matches.to_csv("../data/features/match_features_multi_season.csv", index=False)
team_matches.shape

(2280, 24)

In [10]:
team_matches[
    (team_matches["team"].isin(["Arsenal FC", "AFC Bournemouth", "Manchester City FC"])) &
    (team_matches["season"] == "2024")
][["team", "season", "prior_season_position"]].drop_duplicates()

,team,season,prior_season_position
38,AFC Bournemouth,2024,13.0
152,Arsenal FC,2024,2.0
1406,Manchester City FC,2024,1.0
